# RAG_ICH – Notebook 03: Text Chunking

## Objective

This notebook divides the enriched Clinical Practice Guideline into
retrievable text chunks for Retrieval-Augmented Generation (RAG).

Three chunking strategies will be evaluated:

1. Fixed-size chunking
2. Recursive chunking
3. Semantic chunking

The resulting chunks will later be embedded and stored inside
the vector database.

#Stage 1: Load Corpus

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
FILE_PATH = "/content/drive/MyDrive/RAG_ICH/data/processed/enriched_cpg_text.txt"

with open(FILE_PATH, "r", encoding="utf-8") as f:
    corpus = f.read()

print(f"Corpus loaded successfully.")
print(f"Characters: {len(corpus):,}")

# Import Libraries

In [ ]:
# Install LangChain text splitters
!pip install -q langchain langchain-text-splitters

In [ ]:
# ==========================================================
# Import Libraries
# ==========================================================

from google.colab import drive
from pathlib import Path

import pandas as pd

from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter
)

In [ ]:
# ==========================================================
# Corpus Statistics
# ==========================================================

print("=" * 60)
print("CORPUS STATISTICS")
print("=" * 60)

print(f"Characters : {len(corpus):,}")
print(f"Words      : {len(corpus.split()):,}")
print(f"Lines      : {len(corpus.splitlines()):,}")

# Fixed-size Chunking

In [ ]:
# ==========================================================
# Fixed-Size Chunking
# ==========================================================

fixed_splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=500,
    chunk_overlap=100,
    length_function=len
)

fixed_chunks = fixed_splitter.split_text(corpus)

print(f"Number of chunks: {len(fixed_chunks)}")

In [ ]:
print("=" * 60)
print("FIRST FIXED-SIZE CHUNK")
print("=" * 60)

print(fixed_chunks[0])

In [ ]:
fixed_df = pd.DataFrame({
    "chunk_id": range(1, len(fixed_chunks)+1),
    "text": fixed_chunks
})

fixed_df.head()

Save chunk in CSV

In [ ]:
# OUTPUT_FILE = "/content/drive/MyDrive/RAG_ICH/data/chunks/fixed_chunks.csv"

# Path("/content/drive/MyDrive/RAG_ICH/data/chunks").mkdir(
#     parents=True,
#     exist_ok=True
# )

# fixed_df.to_csv(
#     OUTPUT_FILE,
#     index=False,
#     encoding="utf-8"
# )

# print("Fixed-size chunks saved.")

# Recursive Chunking

In [ ]:
# ==========================================================
# Recursive Chunking
# ==========================================================

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=[
        "\n\n",
        "\n",
        ". ",
        "? ",
        "! ",
        "; ",
        ", ",
        " "
    ],
    length_function=len
)

recursive_chunks = recursive_splitter.split_text(corpus)

print(f"Number of chunks: {len(recursive_chunks)}")

In [ ]:
print("=" * 60)
print("FIRST RECURSIVE CHUNK")
print("=" * 60)

print(recursive_chunks[0])

Create a dataframe

In [ ]:
recursive_df = pd.DataFrame({
    "chunk_id": range(1, len(recursive_chunks) + 1),
    "text": recursive_chunks
})

recursive_df.head()

In [ ]:
# OUTPUT_FILE = "/content/drive/MyDrive/RAG_ICH/data/chunks/recursive_chunks.csv"

# recursive_df.to_csv(
#     OUTPUT_FILE,
#     index=False,
#     encoding="utf-8"
# )

# print("Recursive chunks saved.")

# Comparing fixed vs recursive chunks

In [ ]:
print("=" * 60)
print("CHUNKING COMPARISON")
print("=" * 60)

print(f"Fixed-size chunks : {len(fixed_chunks)}")
print(f"Recursive chunks  : {len(recursive_chunks)}")

# Semantic Chunking

In [ ]:
#Install required package
!pip install -q -U langchain-huggingface
!pip install -q sentence-transformers
!pip install -q langchain-experimental

In [ ]:
#Import package
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_experimental.text_splitter import SemanticChunker

In [ ]:
import torch

print("CUDA Available :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
else:
    print("Using CPU")

In [ ]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True}
)

print("Embedding model loaded successfully.")

In [ ]:
# ==========================================================
# Semantic Chunking
# ==========================================================

semantic_splitter = SemanticChunker(
    embedding_model,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=80
)

print("Semantic chunker initialized successfully.")

In [ ]:
# ==========================================================
# Generate Semantic Chunks
# ==========================================================

semantic_chunks = semantic_splitter.split_text(corpus)

print(f"Number of semantic chunks: {len(semantic_chunks)}")

In [ ]:
import langchain_experimental

print(langchain_experimental.__version__)

In [ ]:
# ==========================================================
# Inspect Semantic Chunks
# ==========================================================

for i in range(min(5, len(semantic_chunks))):

    print("=" * 80)
    print(f"Semantic Chunk {i+1}")
    print("=" * 80)

    print(semantic_chunks[i])
    print("\n")

In [ ]:
# ==========================================================
# Create Semantic Chunk DataFrame
# ==========================================================

semantic_df = pd.DataFrame({
    "chunk_id": range(1, len(semantic_chunks) + 1),
    "text": semantic_chunks
})

semantic_df.head()

In [ ]:
# # ==========================================================
# # Save Semantic Chunks
# # ==========================================================

# OUTPUT_FILE = "/content/drive/MyDrive/RAG_ICH/data/chunks/semantic_chunks.csv"

# semantic_df.to_csv(
#     OUTPUT_FILE,
#     index=False,
#     encoding="utf-8"
# )

# print(f"Semantic chunks saved to:\n{OUTPUT_FILE}")

In [ ]:
for i, chunk in enumerate(semantic_chunks[:5], start=1):
    print("=" * 80)
    print(f"Chunk {i}")
    print(f"Characters: {len(chunk)}")
    print("=" * 80)
    print(chunk[:1000])   # first 1000 characters
    print("\n")

# Compare all chunking strategies

In [ ]:
# ==========================================================
# Compare Chunking Strategies
# ==========================================================

def get_chunk_statistics(chunks):

    lengths = [len(chunk) for chunk in chunks]

    return {
        "Number of Chunks": len(chunks),
        "Average Length": round(sum(lengths) / len(lengths), 2),
        "Minimum Length": min(lengths),
        "Maximum Length": max(lengths)
    }

comparison_df = pd.DataFrame([
    get_chunk_statistics(fixed_chunks),
    get_chunk_statistics(recursive_chunks),
    get_chunk_statistics(semantic_chunks)
])

comparison_df.index = [
    "Fixed-size",
    "Recursive",
    "Semantic"
]

comparison_df

In [ ]:
largest_chunk = max(semantic_chunks, key=len)

print(f"Characters: {len(largest_chunk)}")
print("=" * 80)
print(largest_chunk[:2500])   # First 2,500 characters

# Inspect Chunking

In [ ]:
def inspect_chunk(keyword, chunks, method):

    for i, chunk in enumerate(chunks):

        if keyword.lower() in chunk.lower():

            print("=" * 80)
            print(f"{method} Chunk {i+1}")
            print(f"Characters: {len(chunk)}")
            print("=" * 80)

            print(chunk[:1500])   # Show first 1500 characters

            return

    print(f"{keyword} not found in {method}.")

In [ ]:
inspect_chunk("Recommendation 25", fixed_chunks, "Fixed-size")

inspect_chunk("Recommendation 25", recursive_chunks, "Recursive")

inspect_chunk("Recommendation 25", semantic_chunks, "Semantic")